<center>PROGRAMMAZIONE CONCORRENTE IN PYTHON

-- Thread class: ho vari modi per creare un thread in Python

In [1]:
#Istanzia oggetto Thread, passando "callable object" al costruttore della Classe

import threading
def fun():
    print("Thread running")

if __name__ == "__main__":
     #crea thread
     t1 = threading.Thread(target=fun)

     #starting thread
     t1.start()

     #aspetto la terminazione del thread
     t1.join()
     print("finito")

Thread running
finito


Thread running
Thread running


In [1]:
#Crea una classe che estende quella Thread, ridefinendo il metodo run()
import threading

class myThread(threading.Thread):

    def run(self):
        print("Thread running")

if __name__ == "__main__": #se mai lo importerò, evito di stampare queste cose
    #creo thread
    t1 = myThread()

    #starting thread
    t1.start()

    #aspetto la terminazione del thread
    t1.join()

Thread running


Prova su ciclo di vita del thread (join non distrugge)

Meglio copiare questo esempio su un file .py separato, jupyter dà problemi

In [4]:
from threading import Thread

class MyThread(Thread):

    def run(self):
        print("Running myThread...\n")
        print("is_alive?: ", self.is_alive())
    
    def method1(self):
        print("method1 invocato...\n")

m=MyThread()
m.start()
m.join()
m.method1()
print("is_alive?: ", m.is_alive())

Running myThread...

is_alive?:  True
method1 invocato...

is_alive?:  False


-- Daemon Threads: Thread demoni che vengono ingorati ai fini della terminazione del thread master (in _Jupyter_ non funziona)

In [ ]:
#Non daemon-threads

from threading import *
import time

def thread_1():
    for i in range(5):
        print('this is non-daemon thread')
        time.sleep(2)

#creazione thread
T=Thread(target = thread_1) #mi aspettò che tale thread stampi 5 volte

#starting of thread T
T.start()

#main thread stop execution till 5 sec
time.sleep(5)
print('main Thread execution')

--> qui noto che fa tutte le stampe prima di terminare l'esecuzione

In [ ]:
#daemon threads

from threading import *
import time

def thread_1():
    for i in range(5):
        print('this is 👹 thread T')
        time.sleep(2)

#creating a thread
T = Thread(target = thread_1, daemon = True)

#starting of thread T
T.start() #mi aspetto stampi 5 volte
time.sleep(5)
print("this is main Thread")


--> Qui invece (daemon threads) appena il main vuole, cessa l'esecuzione, uccidendo i daemon

ATTENZIONE! Non funziona in Jupyter!

-- Local class --> Si permette così ai thread di avere uno spazio di memoria locale (altrimenti condividono lo spazio di memoria del processo padre)

In [ ]:
import threading
import logging
import random
logging.basicConfig(level=logging.DEBUG,format='(%(threadName)-0s)%(message)s',)
def show(d):
    try:
        val = d.val
    except AttributeError:
        logging.debug('No value yet')
    else:
        logging.debug('value=%s', val)

def f(d):
    show(d)
    d.val = random.randint(1, 100)
    show(d)

if __name__ == '__main__':
    d = threading.local() #<-- !!
    show(d)
    d.val = 999
    show(d)
for i in range(2): #creo 2 thread con start function f
    t = threading.Thread(target=f, args=(d,))
    t.start()

Esempio SENZA threading.local:

In [22]:
import threading
import time

data ={} #globale a ogni thread

def worker(name, value):
    global data
    data['user'] = value #race condition
    time.sleep(1)
    print(f"{name} vede user =", data['user'])

threads = [threading.Thread(target=worker, args=("T1", "Alice")), threading.Thread(target=worker, args=("T2", "Bob")),]

for t in threads:
        t.start()

for t in threads:
        t.join()

T1 vede user = Bob
T2 vede user = Bob


Esempio CON threading.local:

In [23]:
#Esempio CON threading.local

import threading
import time

#storage locale per thread
data = threading.local() #data è locale ad ogni thread

def worker(name, value):
    data.user = value
    time.sleep(2)
    print(f"{name} vede user=", data.user)

threads = [threading.Thread(target=worker, args=("T1", "Alice")), threading.Thread(target=worker, args=("T2", "Bob")),] #creo 2 thread: mi dovrebbero memorizzare rispettivamente Alice e Bob

for t in threads:
    t.start()

for t in threads:
    t.join()

T1 vede user= Alice
T2 vede user= Bob
